# Country Club SQL Case Study — Tier 2

All 13 questions. Q1–Q9 are written for MySQL/phpMyAdmin; their SQLite-compatible equivalents are used here to verify outputs. Q10–Q13 are SQLite queries. Run all cells with the database file in the same folder.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
conn = sqlite3.connect(Path("sqlite_db_pythonsqlite.db"))
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn))

         name
0    Bookings
1  Facilities
2     Members


## Q1: Facilities that charge members

In [2]:
query = 'SELECT name FROM Facilities WHERE membercost > 0;'
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 5


,name
0,Tennis Court 1
1,Tennis Court 2
2,Massage Room 1
3,Massage Room 2
4,Squash Court


## Q2: Count of free facilities

In [3]:
query = 'SELECT COUNT(*) AS free_facilities FROM Facilities WHERE membercost = 0;'
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 1


,free_facilities
0,4


## Q3: Paid facilities below 20% maintenance

In [4]:
query = 'SELECT facid, name, membercost, monthlymaintenance FROM Facilities WHERE membercost > 0 AND membercost < monthlymaintenance * 0.20;'
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 5


,facid,name,membercost,monthlymaintenance
0,0,Tennis Court 1,5.0,200
1,1,Tennis Court 2,5.0,200
2,4,Massage Room 1,9.9,3000
3,5,Massage Room 2,9.9,3000
4,6,Squash Court,3.5,80


## Q4: Facilities with IDs 1 and 5

In [5]:
query = 'SELECT * FROM Facilities WHERE facid IN (1, 5);'
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 2


,facid,name,membercost,guestcost,initialoutlay,monthlymaintenance
0,1,Tennis Court 2,5.0,25,8000,200
1,5,Massage Room 2,9.9,80,4000,3000


## Q5: Cheap versus expensive facilities

In [6]:
query = "SELECT name, monthlymaintenance, CASE WHEN monthlymaintenance > 100 THEN 'expensive' ELSE 'cheap' END AS cost_category FROM Facilities;"
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 9


,name,monthlymaintenance,cost_category
0,Tennis Court 1,200,expensive
1,Tennis Court 2,200,expensive
2,Badminton Court,50,cheap
3,Table Tennis,10,cheap
4,Massage Room 1,3000,expensive
5,Massage Room 2,3000,expensive
6,Squash Court,80,cheap
7,Snooker Table,15,cheap
8,Pool Table,15,cheap


## Q6: Most recent member(s)

In [7]:
query = 'SELECT firstname, surname FROM Members WHERE joindate = (SELECT MAX(joindate) FROM Members);'
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 1


,firstname,surname
0,Darren,Smith


## Q7: Distinct tennis court users

In [8]:
query = "SELECT DISTINCT f.name AS court, (m.firstname || ' ' || m.surname) AS member FROM Bookings b JOIN Facilities f ON b.facid = f.facid JOIN Members m ON b.memid = m.memid WHERE f.name LIKE 'Tennis Court%' AND m.memid <> 0 ORDER BY member;"
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 44


,court,member
0,Tennis Court 1,Anne Baker
1,Tennis Court 2,Anne Baker
2,Tennis Court 2,Burton Tracy
3,Tennis Court 1,Burton Tracy
4,Tennis Court 1,Charles Owen
5,Tennis Court 2,Charles Owen
6,Tennis Court 2,Darren Smith
7,Tennis Court 1,David Farrell
8,Tennis Court 2,David Farrell
9,Tennis Court 2,David Jones


## Q8: Bookings costing over $30 on September 14 (no subquery)

In [9]:
query = "SELECT f.name AS facility, (m.firstname || ' ' || m.surname) AS member, CASE WHEN b.memid = 0 THEN b.slots * f.guestcost ELSE b.slots * f.membercost END AS cost FROM Bookings b JOIN Facilities f ON b.facid = f.facid JOIN Members m ON b.memid = m.memid WHERE b.starttime >= '2012-09-14' AND b.starttime < '2012-09-15' AND (CASE WHEN b.memid = 0 THEN b.slots * f.guestcost ELSE b.slots * f.membercost END) > 30 ORDER BY cost DESC;"
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 12


,facility,member,cost
0,Massage Room 2,GUEST GUEST,320.0
1,Massage Room 1,GUEST GUEST,160.0
2,Massage Room 1,GUEST GUEST,160.0
3,Massage Room 1,GUEST GUEST,160.0
4,Tennis Court 2,GUEST GUEST,150.0
5,Tennis Court 1,GUEST GUEST,75.0
6,Tennis Court 1,GUEST GUEST,75.0
7,Tennis Court 2,GUEST GUEST,75.0
8,Squash Court,GUEST GUEST,70.0
9,Massage Room 1,Jemima Farrell,39.6


## Q9: Same bookings using a subquery

In [10]:
query = "SELECT facility, member, cost FROM (SELECT f.name AS facility, (m.firstname || ' ' || m.surname) AS member, CASE WHEN b.memid = 0 THEN b.slots * f.guestcost ELSE b.slots * f.membercost END AS cost, b.starttime FROM Bookings b JOIN Facilities f ON b.facid = f.facid JOIN Members m ON b.memid = m.memid) AS booking_costs WHERE starttime >= '2012-09-14' AND starttime < '2012-09-15' AND cost > 30 ORDER BY cost DESC;"
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 12


,facility,member,cost
0,Massage Room 2,GUEST GUEST,320.0
1,Massage Room 1,GUEST GUEST,160.0
2,Massage Room 1,GUEST GUEST,160.0
3,Massage Room 1,GUEST GUEST,160.0
4,Tennis Court 2,GUEST GUEST,150.0
5,Tennis Court 1,GUEST GUEST,75.0
6,Tennis Court 1,GUEST GUEST,75.0
7,Tennis Court 2,GUEST GUEST,75.0
8,Squash Court,GUEST GUEST,70.0
9,Massage Room 1,Jemima Farrell,39.6


## Q10: Facilities with revenue below 1000

In [11]:
query = 'SELECT f.name AS facility, ROUND(SUM(CASE WHEN b.memid = 0 THEN b.slots * f.guestcost ELSE b.slots * f.membercost END), 2) AS revenue FROM Facilities f JOIN Bookings b ON f.facid = b.facid GROUP BY f.facid, f.name HAVING SUM(CASE WHEN b.memid = 0 THEN b.slots * f.guestcost ELSE b.slots * f.membercost END) < 1000 ORDER BY revenue;'
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 3


,facility,revenue
0,Table Tennis,180.0
1,Snooker Table,240.0
2,Pool Table,270.0


## Q11: Members and their recommenders

In [12]:
query = "SELECT m.surname || ', ' || m.firstname AS member, r.surname || ', ' || r.firstname AS recommended_by FROM Members m LEFT JOIN Members r ON CAST(m.recommendedby AS INTEGER) = r.memid WHERE m.memid <> 0 ORDER BY m.surname, m.firstname;"
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 30


,member,recommended_by
0,"Bader, Florence","Stibbons, Ponder"
1,"Baker, Anne","Stibbons, Ponder"
2,"Baker, Timothy","Farrell, Jemima"
3,"Boothe, Tim","Rownam, Tim"
4,"Butters, Gerald","Smith, Darren"
5,"Coplin, Joan","Baker, Timothy"
6,"Crumpet, Erica","Smith, Tracy"
7,"Dare, Nancy","Joplette, Janice"
8,"Farrell, David","GUEST, GUEST"
9,"Farrell, Jemima","GUEST, GUEST"


## Q12: Facility usage excluding guests

In [13]:
query = 'SELECT f.name AS facility, COUNT(*) AS bookings, SUM(b.slots) AS slots_used FROM Facilities f JOIN Bookings b ON f.facid = b.facid WHERE b.memid <> 0 GROUP BY f.facid, f.name ORDER BY facility;'
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 9


,facility,bookings,slots_used
0,Badminton Court,344,1086
1,Massage Room 1,421,884
2,Massage Room 2,27,54
3,Pool Table,783,856
4,Snooker Table,421,860
5,Squash Court,195,418
6,Table Tennis,385,794
7,Tennis Court 1,308,957
8,Tennis Court 2,276,882


## Q13: Monthly facility usage excluding guests

In [14]:
query = "SELECT f.name AS facility, strftime('%Y-%m', b.starttime) AS month, COUNT(*) AS bookings, SUM(b.slots) AS slots_used FROM Facilities f JOIN Bookings b ON f.facid = b.facid WHERE b.memid <> 0 GROUP BY f.facid, f.name, strftime('%Y-%m', b.starttime) ORDER BY facility, month;"
result = pd.read_sql_query(query, conn)
print(f"Rows: {len(result)}")
display(result)

Rows: 27


,facility,month,bookings,slots_used
0,Badminton Court,2012-07,51,165
1,Badminton Court,2012-08,132,414
2,Badminton Court,2012-09,161,507
3,Massage Room 1,2012-07,77,166
4,Massage Room 1,2012-08,153,316
5,Massage Room 1,2012-09,191,402
6,Massage Room 2,2012-07,4,8
7,Massage Room 2,2012-08,9,18
8,Massage Room 2,2012-09,14,28
9,Pool Table,2012-07,103,110


In [16]:
conn.close()